<a href="https://colab.research.google.com/github/amaimanwar8-arch/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected. Tables ready:", list(TABLES.keys()))

Connected. Tables ready: ['dim_clients', 'dim_content', 'fact_daily', 'fact_daily_sample', 'fact_query_90d']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
print("""
Unit of analysis: one row = one content item (content_hash_id), for one client, on one
day (report_date) - the grain of fact_content_daily_performance.

Time window: I'm developing against month=2026-03 (a mid-panel month), per the
assignment's warning that the final month (2026-06) is a sealed test window and should
never be used to develop label logic, since it would double as the natural outcome
window of any past-to-future label.

I'll verify this grain and window with real queries in Section 3.
""")


Unit of analysis: one row = one content item (content_hash_id), for one client, on one
day (report_date) - the grain of fact_content_daily_performance.

Time window: I'm developing against month=2026-03 (a mid-panel month), per the
assignment's warning that the final month (2026-06) is a sealed test window and should
never be used to develop label logic, since it would double as the natural outcome
window of any past-to-future label.

I'll verify this grain and window with real queries in Section 3.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
print("""
Feature: gsc_impressions, gsc_clicks, gsc_avg_position - observed signals known before
any decision point, safe to use as model inputs.

Label/proxy: a decline label I'll define from imp_last30 vs imp_prev30 (impressions
dropping month-over-month), built the same way as Notebook 03's reference example -
never from a product decision flag, since none are shipped in this data.

Context: client_hash_id and content_hash_id - join keys only, used for grouping and
client-holdout splits, never as model features since the codes themselves carry no
signal.

Excluded: trend_direction and trend_pct are excluded from Lane 2 work here since (per
Section 2 of my ML-03 notebook) they are the SOURCE of the starter dataset's declining
label - using them as features on a label derived from them would be leakage, reading
the answer off a different column.
""")


Feature: gsc_impressions, gsc_clicks, gsc_avg_position - observed signals known before
any decision point, safe to use as model inputs.

Label/proxy: a decline label I'll define from imp_last30 vs imp_prev30 (impressions
dropping month-over-month), built the same way as Notebook 03's reference example -
never from a product decision flag, since none are shipped in this data.

Context: client_hash_id and content_hash_id - join keys only, used for grouping and
client-holdout splits, never as model features since the codes themselves carry no
signal.

Excluded: trend_direction and trend_pct are excluded from Lane 2 work here since (per
Section 2 of my ML-03 notebook) they are the SOURCE of the starter dataset's declining
label - using them as features on a label derived from them would be leakage, reading
the answer off a different column.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 — Grain check (confirms "one row = one content item, per client, per day")

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows that break the grain (should be 0): {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows that break the grain (should be 0): 0


,report_date,client_hash_id,content_hash_id,n


Query 2 — Row count and date span for your slice

In [ ]:
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

span_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


Query 3 — Availability check with IS TRUE

In [ ]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS rows_with_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,rows_with_impressions
0,9841378,413966.0,3611061.0


In [ ]:
print("""
Query 1 (grain): 0 rows broke the (report_date, client_hash_id, content_hash_id) grain
in March 2026 - confirms one row = one content item, per client, per day.

Query 2 (slice size): 9,841,378 rows, 331,437 unique content items, 55 clients, spanning
exactly 2026-03-01 to 2026-03-31 - matches the claimed slice.

Query 3 (availability): of 9,841,378 rows, only 413,966 (4.2%) have ga4_data_available
IS TRUE, and 3,611,061 (36.7%) have real search impressions (gsc_impressions > 0). Most
rows in this slice have search tracking but not GA4 - this is real evidence behind the
unbalanced-panel limitation, not a guess.
""")


Query 1 (grain): 0 rows broke the (report_date, client_hash_id, content_hash_id) grain
in March 2026 - confirms one row = one content item, per client, per day.

Query 2 (slice size): 9,841,378 rows, 331,437 unique content items, 55 clients, spanning
exactly 2026-03-01 to 2026-03-31 - matches the claimed slice.

Query 3 (availability): of 9,841,378 rows, only 413,966 (4.2%) have ga4_data_available
IS TRUE, and 3,611,061 (36.7%) have real search impressions (gsc_impressions > 0). Most
rows in this slice have search tracking but not GA4 - this is real evidence behind the
unbalanced-panel limitation, not a guess.



In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    ),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_last30,
            SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY AND f.gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_with_impressions,
            MAX(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks END) AS max_daily_clicks
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_last30 >= 50
    )
    SELECT * FROM windowed
""").df()

print(f"{len(features):,} content items with enough March history")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

115,607 content items with enough March history


,client_hash_id,content_hash_id,imp_last30,clk_last30,pos_last30,days_with_impressions,max_daily_clicks
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,65.0,0.0,4.062836,23.0,0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,29.0,1
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,29.0,1
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,8.978086,27.0,0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,30.0,1


In [ ]:
print("""
Feature notes - all knowable at the decision moment (built only from the prior 30-day
window, no future data touched):

1. imp_last30 (search impressions, last 30 days) - knowable because it's a running
   count of what already happened by the decision day; no future data needed.
2. clk_last30 (search clicks, last 30 days) - same reasoning, purely historical.
3. pos_last30 (average search position, last 30 days) - an average of past daily
   positions, known the moment those days have passed.
4. days_with_impressions (how many of the last 30 days had any impressions) - a count
   of past observed activity, available as soon as the window closes.
5. max_daily_clicks (peak single-day clicks in the window) - the max of values that
   already happened; no forward-looking information involved.
""")


Feature notes - all knowable at the decision moment (built only from the prior 30-day
window, no future data touched):

1. imp_last30 (search impressions, last 30 days) - knowable because it's a running
   count of what already happened by the decision day; no future data needed.
2. clk_last30 (search clicks, last 30 days) - same reasoning, purely historical.
3. pos_last30 (average search position, last 30 days) - an average of past daily
   positions, known the moment those days have passed.
4. days_with_impressions (how many of the last 30 days had any impressions) - a count
   of past observed activity, available as soon as the window closes.
5. max_daily_clicks (peak single-day clicks in the window) - the max of values that
   already happened; no forward-looking information involved.



In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Define a simple decline label from imp_last30 vs a synthetic "prior" reference
# (using days_with_impressions as a stand-in signal, since this is March-only data)
features['is_declining'] = (features['imp_last30'] < features['imp_last30'].median()).astype(int)

honest_cols = ['clk_last30', 'pos_last30', 'days_with_impressions', 'max_daily_clicks']
X = features[honest_cols].fillna(0)
y = features['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"HONEST model (no leak) ROC AUC: {honest_auc:.3f}")

# --- THE TRAP: inject a label-derived column on purpose ---
features['leaked_feature'] = features['imp_last30']  # this IS the label's source column!
leak_cols = honest_cols + ['leaked_feature']
X_leak = features[leak_cols].fillna(0)

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
model_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
leak_auc = roc_auc_score(y_te2, model_leak.predict_proba(X_te2)[:, 1])
print(f"LEAKED model (imp_last30 included) ROC AUC: {leak_auc:.3f}  <- suspiciously high")

print(f"\nAUC jumped from {honest_auc:.3f} to {leak_auc:.3f} just by adding the column")
print("the label itself was built from. Deleting leaked_feature and keeping the honest number.")

HONEST model (no leak) ROC AUC: 0.872
LEAKED model (imp_last30 included) ROC AUC: 1.000  <- suspiciously high

AUC jumped from 0.872 to 1.000 just by adding the column
the label itself was built from. Deleting leaked_feature and keeping the honest number.


In [ ]:
del features['leaked_feature']
print("leaked_feature deleted. The honest, trustworthy number for this feature set is:")
print(f"ROC AUC = {honest_auc:.3f} (using clk_last30, pos_last30, days_with_impressions, max_daily_clicks only)")
print("\nWhy the leak happened: is_declining was defined directly from imp_last30 (below-median")
print("split), so including imp_last30 as a feature let the model just read the threshold back")
print("off itself, rather than learning any real pattern.")

leaked_feature deleted. The honest, trustworthy number for this feature set is:
ROC AUC = 0.872 (using clk_last30, pos_last30, days_with_impressions, max_daily_clicks only)

Why the leak happened: is_declining was defined directly from imp_last30 (below-median
split), so including imp_last30 as a feature let the model just read the threshold back
off itself, rather than learning any real pattern.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
print("""
Data limits - what this slice can never tell you:

1. Unbalanced panel: only 55 clients appear in March 2026's data, and history depth
   differs per client - some have much longer tracking than others. A model trained here
   may reflect a handful of long-tracked clients more than the whole client base.

2. GA4 is sparse in this window: only 4.2% of March rows have ga4_data_available IS TRUE
   (Query 3). Engagement-based features would be unreliable or misleading for the vast
   majority of rows in this slice - not because engagement was zero, but because it
   simply wasn't tracked yet for that client at that time.

3. Window overlap risk: because I built features from the SAME March window I used to
   define the label (imp_last30 vs its own median), this exercise deliberately used a
   proxy label for demonstration - a real capstone label needs a genuinely separate
   future window, never metrics computed from the same days as the features.
""")


Data limits - what this slice can never tell you:

1. Unbalanced panel: only 55 clients appear in March 2026's data, and history depth
   differs per client - some have much longer tracking than others. A model trained here
   may reflect a handful of long-tracked clients more than the whole client base.

2. GA4 is sparse in this window: only 4.2% of March rows have ga4_data_available IS TRUE
   (Query 3). Engagement-based features would be unreliable or misleading for the vast
   majority of rows in this slice - not because engagement was zero, but because it
   simply wasn't tracked yet for that client at that time.

3. Window overlap risk: because I built features from the SAME March window I used to
   define the label (imp_last30 vs its own median), this exercise deliberately used a
   proxy label for demonstration - a real capstone label needs a genuinely separate
   future window, never metrics computed from the same days as the features.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.